In [2]:
from typing import Optional
import os
from pathlib import Path
import pprint
import time
import json
from dotenv import load_dotenv
from warnings import filterwarnings

from langchain_core.documents import Document
from unstructured_client import UnstructuredClient
from unstructured_client.models.operations import CreateJobRequest
from unstructured_client.models.operations import DownloadJobOutputRequest
from unstructured_client.models.shared import BodyCreateJob, InputFiles, JobInformation

load_dotenv()
env = os.getenv("ENV", "dev")

root_dir = Path(os.getcwd()).parent.resolve()
data_dir = root_dir / "data" / env
filterwarnings("ignore")

# Ingestion Plan
```
PDF files (data/{env}/pdfs/)
       │
       ▼
PARSE + CHUNK ─── langchain_unstructured.UnstructuredLoader
       │                 partition_via_api=True, strategy="hi_res"
       │                 chunking_strategy="by_title"
       │                 → List[Document]
       ▼
REFINE (optional)─ langchain_text_splitters.RecursiveCharacterTextSplitter
       │                 → split any oversized chunks
       ▼
METADATA ──────── enrich with paper title, section, source file
       │
       ▼
EMBED ─────────── langchain_openai.OpenAIEmbeddings (dense)
       │              pinecone_text.BM25Encoder (sparse)
       ▼
STORE ─────────── pinecone.Index.upsert() (hybrid vectors)
 ```

In [3]:
def run_on_demand_job(
        client: UnstructuredClient,
        input_dir: str,
        job_template_id: Optional[str] = None, 
        job_nodes: Optional[list[dict[str, object]]] = None
) -> tuple[str, list[dict[str, str]]]:
    """Runs an Unstructured on-demand job.

    Arguments:
    - client {UnstructuredClient}: The initialized Unstructured API client to use.
    - input_dir {str}: The directory that contains the input files.
    - job_template_id: {Optional[str]}: If this job is to use a workflow template, the ID of the workflow template to use.
    - job_nodes {Optional[list[dict[str, object]]]}: If this job is to use a custom workflow definition, the list of custom workflow nodes to use.

    Raises:
    - ValueError: If neither the job template ID nor job nodes (and not both) are specified.
        
    Returns:
    - job_id {str}: The ID of the on-demand job.
    - job_input_file_ids {list[str]}: The input file IDs of the on-demand job.
    - job_output_node_files {list[dict[str, str]]}: The output node files of the on-demand job.
    """
    request_data = {}
    files = []

    for filename in os.listdir(input_dir):
        full_path = os.path.join(input_dir, filename)

        # Skip non-files (for example, directories).
        if not os.path.isfile(full_path):
            continue

        files.append(
            (
                InputFiles(
                    content=open(full_path, "rb"),
                    file_name=filename,
                    content_type="application/pdf"
                )
            )
        )

    if job_template_id is not None:
        request_data = json.dumps({"template_id": job_template_id})
    elif job_nodes is not None:
        request_data = json.dumps({"job_nodes": job_nodes})
    else:
        raise ValueError(f"Must specify a job template ID or job nodes (but not both).")
        exit(1)

    response = client.jobs.create_job(
        request=CreateJobRequest(
            body_create_job=BodyCreateJob(
                request_data=request_data,
                input_files=files
            )
        )
    )

    job_id = response.job_information.id
    job_input_file_ids = response.job_information.input_file_ids
    job_output_node_files = response.job_information.output_node_files

    return job_id, job_input_file_ids, job_output_node_files


def poll_for_job_status(client: UnstructuredClient, job_id: str) -> JobInformation:
    """Keeps checking a job's status until the job is completed.

    Arguments:
    - client {UnstructuredClient}: The initialized Unstructured API client to use.
    - job_id {str}: The job ID to check the status of.

    Returns:
    - job {JobInformation}: Information about the Unstructured job.
    """
    while True:
        response = client.jobs.get_job(
            request={
                "job_id": job_id
            }
        )

        job = response.job_information

        if job.status == "SCHEDULED":
            print("Job is scheduled, polling again in 10 seconds...")
            time.sleep(10)
        elif job.status == "IN_PROGRESS":
            print("Job is in progress, polling again in 10 seconds...")
            time.sleep(10)
        else:
            print("Job is completed.")
            break

    return job


def download_job_output(
        client: UnstructuredClient,
        job_id: str,
        job_input_file_ids: list[str],
        output_dir: str
) -> None:
    """Downloads the output of an Unstructured job.

    Arguments:
    - client {UnstructuredClient}: The initialized Unstructured API client to use.
    - job_id {str}: The job ID to download the output from.
    - job_input_file_ids {list[str]}: The input file IDs of the job.
    - output_dir {str}: The directory to download the output into.
    """
    for job_input_file_id in job_input_file_ids:
        print(f"Attempting to get processed results from file_id '{job_input_file_id}'...")

        response = client.jobs.download_job_output(
            request=DownloadJobOutputRequest(
                job_id=job_id,
                file_id=job_input_file_id
            )
        )

        output_path = os.path.join(output_dir, f"{job_input_file_id}.json")

        with open(output_path, "w") as f:
            json.dump(response.any, f, indent=4)

        print(f"Saved output for file_id '{job_input_file_id}' to '{output_path}'.\n")



In [ ]:
def main():
    # API key and source/destination folder paths.
    UNSTRUCTURED_API_KEY = os.getenv("UNSTRUCTURED_API_KEY")
    INPUT_FOLDER_PATH = data_dir / "pdfs"
    OUTPUT_FOLDER_PATH = data_dir / "parsed"

    # On-demand job settings.
    job_template_id = ""
     
    
    job_nodes = [
            {
                "name": "Partitioner",
                "type": "partition",
                "subtype": "vlm",
                "settings": {
                        "is_dynamic": True,
                        "exclude_elements": ["Header"],
            },
            },
            {
                "name": "Chunker",
                "type": "chunk",
                "subtype": "chunk_by_title",
                "settings": {
                        "include_orig_elements": False,
                        "max_characters": 2048,
                        "multipage_sections": True,
                        "new_after_n_chars": 1500,
                        "overlap": 160,
                        "overlap_all": False,
                        "contextual_chunking_strategy": "v1"
            },
            },
            {
                "name": "Table to HTML",
                "type": "prompter",
                "subtype": "anthropic_table2html",
                "settings": {
                        "model": "claude-sonnet-4-20250514"
            },
            }
        ]
    # Internal tracking variables.
    job_id = ""
    job_input_file_ids = []
    job_output_node_files = []

    with UnstructuredClient(api_key_auth=UNSTRUCTURED_API_KEY) as client:
        print("-" * 80)
        print(
            f"Attempting to run the on-demand job, ingesting the input files from '{INPUT_FOLDER_PATH}'..."
        )
        job_id, job_input_file_ids, job_output_node_files = run_on_demand_job(
            client=client,
            input_dir=INPUT_FOLDER_PATH,
            job_nodes=job_nodes,
        )

        print(f"Job ID: {job_id}\n")
        print("Input file details:\n")

        for job_input_file_id in job_input_file_ids:
            print(job_input_file_id)

        print("\nOutput node file details:\n")

        for output_node_file in job_output_node_files:
            print(output_node_file)

        print("-" * 80)
        print("Polling for job status...")

        job = poll_for_job_status(client, job_id)

        print(f"Job details:\n---\n{job.model_dump_json(indent=4)}")

        if job.status != "COMPLETED":
            print(
                "Job did not complete successfully. Stopping this script without downloading any output."
            )
            exit(1)

        print("-" * 80)
        print("Attempting to download the job output...")
        download_job_output(client, job_id, job_input_file_ids, OUTPUT_FOLDER_PATH)

        print("-" * 80)
        print(
            f"Script completed. Check the output folder '{OUTPUT_FOLDER_PATH}' for the results."
        )
        exit(0)


In [ ]:
main()

--------------------------------------------------------------------------------
Attempting to run the on-demand job, ingesting the input files from 'C:\Users\nikhi\Documents\Projects\HybridRAG-Bench\data\dev\pdfs'...


INFO: HTTP Request: POST https://platform.unstructuredapp.io/api/v1/jobs/ "HTTP/1.1 200 OK"
INFO: HTTP Request: GET https://platform.unstructuredapp.io/api/v1/jobs/9e7538ae-f697-4762-9b1f-ed2731558b49 "HTTP/1.1 200 OK"


Job ID: 9e7538ae-f697-4762-9b1f-ed2731558b49

Input file details:

Attention-Is-All-You-Need_170603762v7-38af4842.pdf

Output node file details:

file_id='Attention-Is-All-You-Need_170603762v7-38af4842.pdf' node_id='a0695262-2798-4f44-b18e-eeee60ff6cb1' node_subtype='vlm' node_type='partition'
file_id='Attention-Is-All-You-Need_170603762v7-38af4842.pdf' node_id='f24cf3fd-e517-40b4-91fb-7b016758990c' node_subtype='chunk_by_title' node_type='chunk'
file_id='Attention-Is-All-You-Need_170603762v7-38af4842.pdf' node_id='011f180a-9efc-421d-8fbb-249177b622b0' node_subtype='anthropic_table2html' node_type='prompter'
--------------------------------------------------------------------------------
Polling for job status...
Job is scheduled, polling again in 10 seconds...


INFO: HTTP Request: GET https://platform.unstructuredapp.io/api/v1/jobs/9e7538ae-f697-4762-9b1f-ed2731558b49 "HTTP/1.1 200 OK"


Job is in progress, polling again in 10 seconds...


...

INFO: HTTP Request: GET https://platform.unstructuredapp.io/api/v1/jobs/9e7538ae-f697-4762-9b1f-ed2731558b49 "HTTP/1.1 200 OK"


Job is in progress, polling again in 10 seconds...


INFO: HTTP Request: GET https://platform.unstructuredapp.io/api/v1/jobs/9e7538ae-f697-4762-9b1f-ed2731558b49 "HTTP/1.1 200 OK"
INFO: HTTP Request: GET https://platform.unstructuredapp.io/api/v1/jobs/9e7538ae-f697-4762-9b1f-ed2731558b49/download?file_id=Attention-Is-All-You-Need_170603762v7-38af4842.pdf "HTTP/1.1 200 OK"


Job is completed.
Job details:
---
{
    "created_at": "2026-03-04T07:47:57.651388Z",
    "id": "9e7538ae-f697-4762-9b1f-ed2731558b49",
    "status": "COMPLETED",
    "workflow_id": "b4054b69-671e-4dbf-b434-45c82d3418c3",
    "workflow_name": "Job 9e7538ae",
    "input_file_ids": [
        "Attention-Is-All-You-Need_170603762v7-38af4842.pdf"
    ],
    "job_type": "ephemeral",
    "output_node_files": [
        {
            "file_id": "Attention-Is-All-You-Need_170603762v7-38af4842.pdf",
            "node_id": "a0695262-2798-4f44-b18e-eeee60ff6cb1",
            "node_subtype": "vlm",
            "node_type": "partition"
        },
        {
            "file_id": "Attention-Is-All-You-Need_170603762v7-38af4842.pdf",
            "node_id": "f24cf3fd-e517-40b4-91fb-7b016758990c",
            "node_subtype": "chunk_by_title",
            "node_type": "chunk"
        },
        {
            "file_id": "Attention-Is-All-You-Need_170603762v7-38af4842.pdf",
            "node_id": "011f180a-

: 